In [ ]:
# ============ 1. SETTINGS ============
DATA_DIR = "../data/combined"   # folder holding classes.json + train/ val/ test/
ALPHA = 0.45                                                  # overlay opacity
STEP = 2                                                      # display downsampling; 1 = full 2560x2560 (slow)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap, to_hex, to_rgb
from matplotlib.patches import Patch
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
ROOT = Path(DATA_DIR)
IGNORE = 255              # unannotated pixels; not listed in classes.json
IGNORE_COLOR = "#ff00ff"

# classes.json gives id -> label (these ids are what the mask pixels contain).
# Colors are assigned here rather than read from the export, so labels that
# repeat (0 and 1 both "background") still get distinguishable shades.
LABELS = {int(k): v for k, v in json.loads((ROOT / "classes.json").read_text()).items()}
PALETTE = {i: to_hex(plt.get_cmap("tab20")(n % 20)) for n, i in enumerate(sorted(LABELS))}

name = lambda i: "ignore/unannotated" if i == IGNORE else LABELS.get(int(i), f"UNDECLARED {i}")
color = lambda i: IGNORE_COLOR if i == IGNORE else PALETTE.get(int(i), "#000000")
CMAP = ListedColormap([to_rgb(color(i)) for i in range(256)])

# pair up images/ with same-named files in the sibling masks/,
# skipping non-images and hidden junk like .DS_Store
EXTS = {".png", ".tif", ".tiff", ".jpg", ".jpeg"}
is_image = lambda p: p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith(".")
PAIRS = [
    (img, d.parent / "masks" / img.name)
    for d in sorted(p for p in ROOT.rglob("images") if p.is_dir())
    for img in sorted(f for f in d.iterdir() if is_image(f))
    if is_image(d.parent / "masks" / img.name)
]
read = lambda p: (lambda a: a[..., 0] if a.ndim == 3 else a)(np.asarray(Image.open(p)))

print(f"{ROOT.name}: {len(PAIRS)} image/mask pairs, ignore = {IGNORE}")
for i in sorted(LABELS):
    dup = "  (label repeats)" if list(LABELS.values()).count(LABELS[i]) > 1 else ""
    print(f"  {i}: {LABELS[i]:<12} {color(i)}{dup}")
print(f"  {IGNORE}: ignore/unannotated  {IGNORE_COLOR}   <- exclude from loss")

In [ ]:
# ============ 2. PLOT EVERY IMAGE / MASK PAIR ============
for img_path, mask_path in PAIRS:
    image, mask = read(img_path)[::STEP, ::STEP], read(mask_path)[::STEP, ::STEP]
    present = [int(v) for v in np.unique(mask)]
    hide = mask == IGNORE

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.8))
    axes[0].imshow(image, cmap="gray" if image.ndim == 2 else None)
    axes[0].set_title("image")
    axes[1].imshow(mask, cmap=CMAP, vmin=0, vmax=255, interpolation="nearest")
    axes[1].set_title("mask")
    axes[2].imshow(image, cmap="gray" if image.ndim == 2 else None)
    axes[2].imshow(np.ma.masked_where(hide, mask), cmap=CMAP, vmin=0, vmax=255,
                   alpha=ALPHA, interpolation="nearest")
    axes[2].set_title(f"overlay (alpha={ALPHA}, ignore transparent)")
    for ax in axes:
        ax.axis("off")
    axes[2].legend(handles=[Patch(facecolor=color(i), label=f"{i}  {name(i)}") for i in present],
                   loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    fig.suptitle(f"{img_path.parent.parent.name} / {img_path.name}", fontsize=11)
    fig.tight_layout()
    plt.show()

In [ ]:
# ============ 3. DISTINCT VALUES IN EACH MASK ============
# A mask is a 2-D uint8 array where the pixel value IS the class id from classes.json.
for img_path, mask_path in PAIRS:
    mask = read(mask_path)
    ids, counts = np.unique(mask, return_counts=True)
    print(f"{img_path.parent.parent.name}/{mask_path.name}  {mask.dtype}{mask.shape}  "
          f"{len(ids)} distinct values: {[int(i) for i in ids]}")
    for i, c in zip(ids, counts):
        print(f"      {int(i):>3}  {name(i):<20} {c:>12,}  {c / mask.size:>7.2%}")
    never = [f"{i}:{LABELS[i]}" for i in sorted(LABELS) if i not in ids]
    if never:
        print(f"      never painted: {', '.join(never)}")
    print()

print(f"values used across dataset : {sorted({int(v) for _, m in PAIRS for v in np.unique(read(m))})}")
print(f"ids declared in classes.json: {sorted(LABELS)}")
print(f"ignore_index                : {IGNORE}")

# raw numbers from one mask, downsampled so the grid fits on screen
mask = read(PAIRS[0][1])
step = max(mask.shape) // 40 or 1
np.set_printoptions(linewidth=200, threshold=100_000)
print(f"\n{PAIRS[0][1].name}, every {step}th pixel:")
print(mask[::step, ::step])

## 